# Phase 13 Model Testing Framework

## Interactive Testing for 9D NLP Policy with Modified Reward Structure

This notebook runs comprehensive tests on the Phase 13 PPO model:
- **Observation**: 9D NLP intent vector (vs Phase 10 raw state)
- **Reward**: Potential-based shaping with turn penalties
- **Action Masking**: Escalate blocked for turn_count < 3
- **Curriculum**: Progressive subflow unlocking
- **LLMs**: Intent classification + Agent response generation

## Setup & Configuration

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import json
import time

# Add repo to path
repo_root = Path('.').resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

from stable_baselines3 import PPO
from Simulation_4.training.reward_shaping import RewardShaper

print("✓ All imports successful")

Repo root: /Users/tishabhavsar/RL_Project 


ModuleNotFoundError: No module named 'Simulation_4'

In [ ]:
# Paths
ARTIFACTS_ROOT = "Simulation_4/artifacts"
MODEL_PATH = "Simulation_4/artifacts/phase10_prod/models/best_model.zip"

print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Model path: {MODEL_PATH}")
print(f"Model exists: {Path(MODEL_PATH).exists()}")

---
# TEST 1: Model Architecture Validation

## What it checks:
- ✓ Model loads successfully
- ✓ Observation space is 9D (not some other dimension)
- ✓ Action space is Discrete(5)
- ✓ Network is MLP with [64, 32] architecture

## Why it matters:
Phase 13 uses a smaller, semantically-rich 9D observation space (vs Phase 10's larger raw state). If this dimension is wrong, downstream inference will fail.

In [ ]:
print("="*70)
print("TEST 1: Model Architecture Validation")
print("="*70)

try:
    model = PPO.load(MODEL_PATH)
    print(f"✓ Model loaded successfully")
    
    obs_space = model.observation_space
    action_space = model.action_space
    
    obs_size = obs_space.shape[0] if hasattr(obs_space, 'shape') else None
    action_size = action_space.n if hasattr(action_space, 'n') else None
    
    policy_type = type(model.policy).__name__
    net_arch = model.policy_kwargs.get('net_arch', 'unknown')
    
    print(f"\n  Policy type: {policy_type}")
    print(f"  Observation space: {obs_space}")
    print(f"    → Size: {obs_size} (expected 9 for Phase 13)")
    print(f"  Action space: {action_space}")
    print(f"    → Size: {action_size} (expected 5)")
    print(f"  Network: {net_arch}")
    print(f"    → Expected: [64, 32] for Phase 13")
    
    obs_match = obs_size == 9
    action_match = action_size == 5
    
    if obs_match:
        print("\n  ✓ Observation space matches Phase 13 (9D NLP)")
    else:
        print(f"\n  ✗ Observation mismatch: got {obs_size}, expected 9")
        
    if action_match:
        print("  ✓ Action space matches Phase 13 (Discrete(5))")
    else:
        print(f"  ✗ Action mismatch: got {action_size}, expected 5")
    
    TEST_1_PASSED = obs_match and action_match
    print(f"\n  => TEST 1: {'PASSED ✓' if TEST_1_PASSED else 'FAILED ✗'}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    TEST_1_PASSED = False

---
# TEST 2: Reward Structure Validation

## What it checks:
- ✓ RewardShaper is instantiated correctly
- ✓ Strict potential mode is enabled (policy-invariant)
- ✓ Potential function computes correctly
- ✓ Reward shaping formula applies: `reward_new = base_reward + gamma*Φ(s') - Φ(s)`

## Why it matters:
Phase 13 uses a modified reward structure with **potential-based shaping**. This encourages faster learning without changing the optimal policy. Incorrect shaping can lead to suboptimal policies.

In [ ]:
print("="*70)
print("TEST 2: Reward Structure Validation")
print("="*70)

try:
    shaper = RewardShaper(enabled=True, strict_potential=True)
    
    print(f"✓ RewardShaper instantiated")
    print(f"\n  Configuration:")
    print(f"    Enabled: {shaper.enabled}")
    print(f"    Strict potential mode: {shaper.strict_potential}")
    print(f"    Info gain bonus: {shaper.info_gain_bonus}")
    print(f"    Progress increase bonus: {shaper.progress_increase_bonus}")
    print(f"    Frustration decrease bonus: {shaper.frustration_decrease_bonus}")
    print(f"    Gamma (discount): {shaper.gamma}")
    
    # Test potential function
    test_state_pre = {"information": 0.3, "progress": 0.2, "frustration": 0.5}
    test_state_post = {"information": 0.6, "progress": 0.5, "frustration": 0.2}
    
    phi_pre = shaper._potential(test_state_pre)
    phi_post = shaper._potential(test_state_post)
    
    print(f"\n  Potential function test:")
    print(f"    Pre-state potential: {phi_pre:.4f}")
    print(f"    Post-state potential: {phi_post:.4f}")
    
    # Test shaping
    shaped = shaper.shape(1.0, test_state_pre, test_state_post, 0, {})
    print(f"    Shaped reward (base=1.0): {shaped:.4f}")
    print(f"    Shaping delta: {shaped - 1.0:.4f}")
    
    TEST_2_PASSED = shaper.enabled and shaper.strict_potential
    print(f"\n  => TEST 2: {'PASSED ✓' if TEST_2_PASSED else 'FAILED ✗'}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    TEST_2_PASSED = False

---
# TEST 3: Phase 10 vs Phase 13 Comparison

## Key Architectural Changes:

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Component': [
        'Observation Space',
        'Observation Source',
        'Reward Logic',
        'Action Safety',
        'Agent Responses',
        'Training Strategy',
        'Network Size'
    ],
    'Phase 10': [
        'Raw state (unclear dims)',
        'Direct StateEngine',
        'Base reward only',
        'No constraints',
        'Templates/Fixed',
        'Fixed subflows',
        'Larger network'
    ],
    'Phase 13 (NEW)': [
        '9D NLP intent vector ✓',
        'IntentClassifier LLM ✓',
        'Potential-based shaping ✓',
        'Escalate masked < turn 3 ✓',
        'LLM-generated ✓',
        'Curriculum learning ✓',
        '[64, 32] Tanh optimized ✓'
    ]
})

print("\n" + "="*70)
print("Phase 10 vs Phase 13 - Key Changes")
print("="*70 + "\n")
print(comparison.to_string(index=False))
print()

---
# SUMMARY

In [ ]:
print("\n" + "█"*70)
print("  PHASE 13 MODEL TEST SUMMARY")
print("█"*70)

tests = {
    "TEST 1 - Model Architecture": TEST_1_PASSED,
    "TEST 2 - Reward Structure": TEST_2_PASSED,
}

passed = sum(tests.values())
total = len(tests)

print(f"\nTests passed: {passed}/{total}\n")

for test_name, result in tests.items():
    status = "✓ PASSED" if result else "✗ FAILED"
    print(f"  {test_name:.<50} {status}")

print(f"\n" + "█"*70)

if passed == total:
    print("\n🎉 ALL TESTS PASSED!\n")
    print("Phase 13 model is functioning correctly with:")
    print("  ✓ 9D NLP observation space")
    print("  ✓ Potential-based reward shaping")
    print("  ✓ Modified reward structure")
    print()
else:
    print(f"\n⚠️  {total - passed} test(s) failed\n")

## Next Steps:

1. **If all tests pass:**
   - Model is correctly structured
   - Ready for inference/evaluation
   - Compare with Phase 10 baseline

2. **If tests fail:**
   - Check error messages above
   - Review Phase 13 training logs
   - Verify model path is correct

3. **For detailed analysis:**
   - Run batch evaluation (25+ episodes)
   - Check resolution/escalation/dropout rates
   - Compare reward distributions